# BMW Repair Order — Trace + GEPA Prompt Optimization

**Pipeline:**
1. Surya OCR extracts layout-aware text from PDF (runs once, cached by MD5)
2. Gemini 2.0 Flash extracts structured JSON from OCR text
3. `eval.py` scores extraction against ground-truth JSON
4. GEPA-style reflection loop iteratively rewrites the system prompt
5. Pareto frontier tracks best non-dominated prompts across all 6 docs

## 1. Setup

In [ ]:
# Run once if packages are missing
!pip install -q google-generativeai PyMuPDF deepdiff
# surya-ocr and pdf2image removed — PDFs have a native text layer; PyMuPDF extracts it directly

In [ ]:
import os, json, copy, time, hashlib, pickle, re, sys
from pathlib import Path
from dataclasses import dataclass, field, asdict
import numpy as np
import matplotlib.pyplot as plt
import google.generativeai as genai

# ── Paths ────────────────────────────────────────────────────────────────────

def _find_repo_root(start: Path) -> Path:
    """Walk up until we find a directory containing Data/Samples/ and Scripts/eval.py."""
    for p in [start, start.parent, start.parent.parent, start.parent.parent.parent]:
        if (p / "Data" / "Samples").exists() and (p / "Scripts" / "eval.py").exists():
            return p
    return start

BASE_DIR   = _find_repo_root(Path().resolve())
DATA_DIR   = BASE_DIR / "Data" / "Samples"
SCHEMA_PATH = BASE_DIR / "Data" / "extraction_schema.json"
SCRIPTS_DIR = BASE_DIR / "Scripts"
OCR_CACHE   = BASE_DIR / "Experimentation" / "Ivy" / "ocr_cache.pkl"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"Schema     : {SCHEMA_PATH.exists()}")
print(f"eval.py    : {(SCRIPTS_DIR / 'eval.py').exists()}")

# ── Eval import ──────────────────────────────────────────────────────────────
sys.path.insert(0, str(SCRIPTS_DIR))
from eval import evaluate_extraction

# ── API key ──────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.0-flash")
print("Gemini model ready.")

# ── Schema ───────────────────────────────────────────────────────────────────
with open(SCHEMA_PATH) as f:
    SCHEMA_STR = json.dumps(json.load(f), indent=2)

# ── Auto-discover PDF/JSON pairs ─────────────────────────────────────────────
PDF_FILES: dict[str, Path] = {}
GT_FILES:  dict[str, dict] = {}

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    doc_id   = pdf_path.stem
    json_path = DATA_DIR / f"{doc_id}.json"
    if json_path.exists():
        PDF_FILES[doc_id] = pdf_path
        with open(json_path) as f:
            GT_FILES[doc_id] = json.load(f)

print(f"\nFound {len(PDF_FILES)} PDF/JSON pairs: {sorted(PDF_FILES.keys())}")

## 2. OCR Intake (with MD5 caching)

In [ ]:
import fitz  # PyMuPDF

def pdf_to_text(pdf_path) -> str:
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n--- PAGE BREAK ---\n\n".join(pages)

# ── Cache load ────────────────────────────────────────────────────────────────
OCR_CACHE.parent.mkdir(parents=True, exist_ok=True)
ocr_cache: dict = {}
if OCR_CACHE.exists():
    with open(OCR_CACHE, "rb") as f:
        ocr_cache = pickle.load(f)
    print(f"Loaded OCR cache: {len(ocr_cache)} entries.")

# ── Pre-run text extraction for all docs ─────────────────────────────────────
ocr_texts: dict[str, str] = {}

for doc_id, pdf_path in sorted(PDF_FILES.items()):
    md5 = hashlib.md5(pdf_path.read_bytes()).hexdigest()
    if md5 in ocr_cache:
        print(f"  {doc_id}: cache hit")
        ocr_texts[doc_id] = ocr_cache[md5]
    else:
        print(f"  {doc_id}: extracting text...", end=" ", flush=True)
        text = pdf_to_text(pdf_path)
        ocr_texts[doc_id] = text
        ocr_cache[md5]    = text
        with open(OCR_CACHE, "wb") as f:
            pickle.dump(ocr_cache, f)
        print(f"done ({len(text):,} chars)")

print(f"\nText extraction complete for {len(ocr_texts)} docs.")

# Quick sanity check
sample_id = sorted(ocr_texts.keys())[0]
print(f"\nSample output — {sample_id} (first 500 chars):")
print(ocr_texts[sample_id][:500])

## 3. Extraction Function

In [ ]:
MODEL_ID    = "gemini-2.0-flash"
TEMPERATURE = 0.0  # deterministic — score changes must come from prompt changes, not sampling


def extract_json(system_prompt: str, ocr_text: str, retries: int = 2) -> dict:
    """
    Call Gemini with the given system prompt + OCR text.
    Returns a parsed dict or {"_extraction_error": str} on failure.
    """
    user_msg = (
        "Here is the repair order text extracted via OCR:\n\n"
        + ocr_text
        + "\n\nExtract the complete structured JSON according to the schema."
    )
    full_prompt = system_prompt + "\n\n" + user_msg

    gen_cfg = genai.GenerationConfig(
        temperature=TEMPERATURE,
        response_mime_type="application/json",
    )

    for attempt in range(retries + 1):
        try:
            response = model.generate_content(full_prompt, generation_config=gen_cfg)
            raw = response.text.strip()
            # Strip markdown code fences if the model returns them anyway
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            return json.loads(raw)
        except Exception as exc:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {"_extraction_error": str(exc)}

## 4. Evaluation Wrapper

In [ ]:
def evaluate(prediction: dict, doc_id: str) -> dict:
    """
    Wrap evaluate_extraction from eval.py.
    Returns {"score": float, "issues": list, "subscores": dict}.
    """
    if "_extraction_error" in prediction:
        return {
            "score": 0.0,
            "issues": [{"detail": prediction["_extraction_error"],
                        "category": "structure", "kind": "missing",
                        "path": "", "penalty": 1.0, "severity": "high"}],
            "subscores": {"structure": 0.0, "numbers": 0.0, "text": 0.0},
        }

    gt = GT_FILES[doc_id]
    try:
        result = evaluate_extraction(gt, prediction)
        score  = min(1.0, float(result["score"]))  # defensive cap
        return {
            "score":    score,
            "issues":   result.get("issues", []),
            "subscores": result.get("subscores", {}),
        }
    except Exception as exc:
        return {
            "score": 0.0,
            "issues": [{"detail": str(exc), "category": "structure",
                        "kind": "missing", "path": "", "penalty": 1.0, "severity": "high"}],
            "subscores": {"structure": 0.0, "numbers": 0.0, "text": 0.0},
        }

## 5. Trace Data Structures

In [ ]:
@dataclass
class Trace:
    iteration:     int
    doc_id:        str
    prompt_id:     str
    system_prompt: str
    ocr_text:      str
    prediction:    dict
    score:         float
    issues:        list
    subscores:     dict


@dataclass
class PromptCandidate:
    prompt_id:     str
    system_prompt: str
    iteration:     int
    traces: list = field(default_factory=list)
    scores: list = field(default_factory=list)

    @property
    def mean_score(self) -> float:
        return float(np.mean(self.scores)) if self.scores else 0.0

    @property
    def min_score(self) -> float:
        return float(np.min(self.scores)) if self.scores else 0.0

    def is_pareto_dominated_by(self, other: "PromptCandidate") -> bool:
        """
        Returns True if `other` weakly dominates self on all docs
        and strictly dominates on at least one.
        """
        if not self.traces or not other.traces:
            return False
        self_by_doc  = {t.doc_id: t.score for t in self.traces}
        other_by_doc = {t.doc_id: t.score for t in other.traces}
        common = set(self_by_doc) & set(other_by_doc)
        if not common:
            return False
        all_geq = all(other_by_doc[d] >= self_by_doc[d] for d in common)
        any_gt  = any(other_by_doc[d] >  self_by_doc[d] for d in common)
        return all_geq and any_gt


# ── Global state ──────────────────────────────────────────────────────────────
all_traces:        list[Trace]           = []
pareto_frontier:   list[PromptCandidate] = []
iteration_history: list[dict]            = []
_prompt_counter    = [0]


def _new_prompt_id() -> str:
    _prompt_counter[0] += 1
    return f"P{_prompt_counter[0]:03d}"


def _prompt_hash(prompt: str) -> str:
    return hashlib.md5(prompt.encode()).hexdigest()


def run_prompt_on_all_docs(system_prompt: str, iteration: int) -> PromptCandidate:
    """Evaluate system_prompt on every doc; return a populated PromptCandidate."""
    prompt_id = _new_prompt_id()
    candidate = PromptCandidate(
        prompt_id=prompt_id,
        system_prompt=system_prompt,
        iteration=iteration,
    )

    for doc_id in sorted(PDF_FILES.keys()):
        print(f"  [{prompt_id}] {doc_id}...", end=" ", flush=True)
        prediction  = extract_json(system_prompt, ocr_texts[doc_id])
        eval_result = evaluate(prediction, doc_id)

        trace = Trace(
            iteration=iteration,
            doc_id=doc_id,
            prompt_id=prompt_id,
            system_prompt=system_prompt,
            ocr_text=ocr_texts[doc_id],
            prediction=prediction,
            score=eval_result["score"],
            issues=eval_result["issues"],
            subscores=eval_result["subscores"],
        )
        candidate.traces.append(trace)
        candidate.scores.append(eval_result["score"])
        all_traces.append(trace)
        print(f"score={eval_result['score']:.3f}")

    print(f"  → mean={candidate.mean_score:.3f}  min={candidate.min_score:.3f}")
    return candidate


print("Data structures defined.")

## 6. Baseline Run

In [ ]:
INITIAL_SYSTEM_PROMPT = f"""You are an expert data extractor for BMW automotive repair orders.
Your task is to read OCR text from a repair order and return a single, complete JSON object.

## Output Format
Return ONLY valid JSON — no markdown fences, no explanation, no commentary.

## Schema (your output must conform exactly)

{SCHEMA_STR}

## Extraction Rules
1. `doc_id`: the repair order (RO) number as a string.
2. `sections`: an array of all sections found in the document. Section prefixes include: ASI, BWO, CSI, JSI, WSI, ISI.
3. `header`: extract every field listed in the schema. Use null for any field not present in the document.
4. `footer`: extract all financial totals (labor_amount, parts_amount, gas_oil_lube, sublet_amount, misc_charges, total_charges, less_insurance, sales_tax, please_pay). Use 0.0 if a field is explicitly zero; null if absent.
5. `content.job`: extract each repair job as an object. Capture all nested ops (operations), parts, labor, and log arrays.
6. Numeric fields must be numbers (int or float), never strings. Dates as strings in MM/DD/YYYY format. Times as HH:MM.
7. Do not invent or hallucinate values — only extract what is present in the OCR text.

Output only the JSON object.
"""

evaluated_hashes = {_prompt_hash(INITIAL_SYSTEM_PROMPT)}

print("Running baseline (iteration 0)...")
baseline_candidate = run_prompt_on_all_docs(INITIAL_SYSTEM_PROMPT, iteration=0)
pareto_frontier.append(baseline_candidate)
print(f"\nBaseline complete. Mean={baseline_candidate.mean_score:.3f}  Min={baseline_candidate.min_score:.3f}")

## 7. GEPA Reflection Prompt Builder

In [ ]:
def build_reflection_prompt(frontier: list[PromptCandidate], iteration: int) -> str:
    """Construct the meta-prompt that asks Gemini to reflect and propose an improved extractor prompt."""

    # Best frontier candidate by mean score
    best = max(frontier, key=lambda c: c.mean_score)

    # Worst trace per doc (worst score ever seen across all iterations)
    worst_by_doc: dict[str, Trace] = {}
    for t in all_traces:
        if t.doc_id not in worst_by_doc or t.score < worst_by_doc[t.doc_id].score:
            worst_by_doc[t.doc_id] = t
    worst_traces = sorted(worst_by_doc.values(), key=lambda t: t.score)

    # Build failure analysis block
    failure_blocks = []
    for t in worst_traces:
        top_issues = t.issues[:10]
        issue_lines = "\n".join(
            f"    [{i.get('severity','?')}] {i.get('category','?')}/{i.get('kind','?')} "
            f"@ {i.get('path','?')!r}  expected={str(i.get('expected',''))[:50]!r}  "
            f"got={str(i.get('got',''))[:50]!r}"
            for i in top_issues
        ) or "    (no issues logged)"
        sub = t.subscores
        sample = json.dumps(t.prediction, indent=2)[:600] if t.prediction else "N/A"
        failure_blocks.append(
            f"Doc {t.doc_id}  score={t.score:.3f}  prompt={t.prompt_id}  iter={t.iteration}\n"
            f"  subscores: structure={sub.get('structure',0):.2f}  "
            f"numbers={sub.get('numbers',0):.2f}  text={sub.get('text',0):.2f}\n"
            f"  top issues:\n{issue_lines}\n"
            f"  sample output (truncated to 600 chars):\n{sample}"
        )

    # Build frontier summary
    frontier_lines = []
    for c in sorted(frontier, key=lambda x: -x.mean_score):
        doc_scores = "  ".join(
            f"{t.doc_id}={t.score:.3f}" for t in sorted(c.traces, key=lambda x: x.doc_id)
        )
        frontier_lines.append(
            f"  [{c.prompt_id}] iter={c.iteration}  mean={c.mean_score:.3f}  "
            f"min={c.min_score:.3f}\n    per-doc: {doc_scores}"
        )

    return f"""You are a prompt engineering expert optimizing a JSON extraction system for BMW automotive repair orders.

== CURRENT BEST SYSTEM PROMPT [{best.prompt_id}  mean={best.mean_score:.3f}] ==

{best.system_prompt}

== EXTRACTION SCHEMA (verbatim) ==

{SCHEMA_STR}

== FAILURE ANALYSIS (worst-performing doc per document ID, worst-first) ==

{chr(10).join(failure_blocks)}

== PARETO FRONTIER ({len(frontier)} candidate(s)) ==

{chr(10).join(frontier_lines)}

== YOUR TASK (Iteration {iteration}) ==

Step 1 — REFLECT: Identify the recurring failure patterns from the analysis above.
Consider: missing fields, wrong numeric types, incorrect nesting, hallucinated values, date/time format errors.

Step 2 — HYPOTHESIZE: For each failure pattern, state a concrete hypothesis about what prompt change would fix it.

Step 3 — PROPOSE: Write a complete, improved system prompt.

Constraints:
- Every added instruction must address a specific observed failure — no vague generalities.
- Embed the full schema verbatim.
- The prompt must produce JSON-only output (no markdown, no explanation).
- Build incrementally on the current best prompt; do not start from scratch.

Output EXACTLY this format:

<reflection>
Your analysis of failure patterns and hypotheses (be specific and concise).
</reflection>

<improved_prompt>
The complete improved system prompt, ready to use as-is.
</improved_prompt>
"""

## 8. GEPA Loop (5 iterations)

In [ ]:
def update_pareto_frontier(frontier: list[PromptCandidate], candidate: PromptCandidate) -> None:
    """Add candidate if not dominated; evict any candidates it dominates."""
    if any(candidate.is_pareto_dominated_by(c) for c in frontier):
        print(f"  [{candidate.prompt_id}] dominated — not added to frontier")
        return
    dominated = [c for c in frontier if c.is_pareto_dominated_by(candidate)]
    for c in dominated:
        print(f"  [{candidate.prompt_id}] evicts dominated [{c.prompt_id}]")
    frontier[:] = [c for c in frontier if c not in dominated]
    frontier.append(candidate)
    print(f"  [{candidate.prompt_id}] added to frontier (size={len(frontier)})")


N_ITERATIONS = 5  # 5 iters × 7 API calls = 35 total (1 reflection + 6 extractions)

reflection_gen_cfg = genai.GenerationConfig(temperature=0.3)  # mild exploration for hypotheses

for iteration in range(1, N_ITERATIONS + 1):
    print(f"\n{'='*64}")
    print(f"  GEPA Iteration {iteration}/{N_ITERATIONS}")
    print(f"{'='*64}")

    # 1. Build reflection prompt
    ref_prompt = build_reflection_prompt(pareto_frontier, iteration)

    # 2. Call Gemini for reflection + improved prompt
    print("Calling Gemini for reflection...")
    ref_response = model.generate_content(ref_prompt, generation_config=reflection_gen_cfg)
    ref_text     = ref_response.text

    # 3. Parse tags
    reflection_m = re.search(r"<reflection>(.*?)</reflection>", ref_text, re.DOTALL)
    improved_m   = re.search(r"<improved_prompt>(.*?)</improved_prompt>", ref_text, re.DOTALL)

    reflection_text = reflection_m.group(1).strip() if reflection_m else ""
    new_prompt      = improved_m.group(1).strip()   if improved_m   else ""

    if not new_prompt:
        print("  WARNING: could not parse <improved_prompt> — skipping iteration.")
        print("  Raw response snippet:", ref_text[:300])
        continue

    print(f"  Reflection (excerpt): {reflection_text[:250]}...")

    # 4. Deduplicate by prompt hash
    ph = _prompt_hash(new_prompt)
    if ph in evaluated_hashes:
        print("  Identical prompt already evaluated — skipping.")
        continue
    evaluated_hashes.add(ph)

    # 5. Evaluate new prompt across all docs
    print(f"  Evaluating on {len(PDF_FILES)} docs...")
    new_candidate = run_prompt_on_all_docs(new_prompt, iteration)

    # 6. Update Pareto frontier
    update_pareto_frontier(pareto_frontier, new_candidate)

    # 7. Log iteration
    iteration_history.append({
        "iteration":         iteration,
        "prompt_id":         new_candidate.prompt_id,
        "mean_score":        new_candidate.mean_score,
        "min_score":         new_candidate.min_score,
        "frontier_size":     len(pareto_frontier),
        "reflection_excerpt": reflection_text[:500],
    })

    best_now = max(pareto_frontier, key=lambda c: c.mean_score)
    print(f"  Best mean so far: {best_now.mean_score:.3f} [{best_now.prompt_id}]")

print("\nGEPA loop complete.")

## 9. Results

In [ ]:
# ── Build per-iteration score table ──────────────────────────────────────────
# Each iteration may have evaluated exactly one new candidate.
# We track the evaluated candidate (not frontier best) for the trajectory plot.

# Collect all candidates in evaluation order (baseline + GEPA candidates)
all_candidates: list[PromptCandidate] = [baseline_candidate]
seen_pids = {baseline_candidate.prompt_id}
for h in iteration_history:
    pid = h["prompt_id"]
    if pid not in seen_pids:
        match = next((c for c in pareto_frontier if c.prompt_id == pid), None)
        if match is None:
            # candidate was not on frontier — reconstruct from all_traces
            traces_for = [t for t in all_traces if t.prompt_id == pid]
            if traces_for:
                c = PromptCandidate(
                    prompt_id=pid,
                    system_prompt=traces_for[0].system_prompt,
                    iteration=h["iteration"],
                    traces=traces_for,
                    scores=[t.score for t in traces_for],
                )
                all_candidates.append(c)
                seen_pids.add(pid)
        else:
            all_candidates.append(match)
            seen_pids.add(pid)

# ── Score trajectory ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_labels = [f"iter {c.iteration}\n{c.prompt_id}" for c in all_candidates]
x_pos    = list(range(len(all_candidates)))
means    = [c.mean_score for c in all_candidates]
mins     = [c.min_score  for c in all_candidates]

ax = axes[0]
ax.plot(x_pos, means, "b-o", label="Mean score", zorder=3)
ax.plot(x_pos, mins,  "r--s", label="Min score",  zorder=3)
ax.axhline(y=baseline_candidate.mean_score, color="gray", linestyle=":",
           alpha=0.6, label="Baseline mean")
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, fontsize=8)
ax.set_ylabel("Score")
ax.set_title("Score Trajectory (evaluated candidates)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Per-doc heatmap ───────────────────────────────────────────────────────────
doc_ids     = sorted(PDF_FILES.keys())
score_matrix = np.full((len(doc_ids), len(all_candidates)), np.nan)

for j, candidate in enumerate(all_candidates):
    by_doc = {t.doc_id: t.score for t in candidate.traces}
    for i, doc_id in enumerate(doc_ids):
        if doc_id in by_doc:
            score_matrix[i, j] = by_doc[doc_id]

ax2 = axes[1]
im  = ax2.imshow(score_matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels, fontsize=8)
ax2.set_yticks(range(len(doc_ids)))
ax2.set_yticklabels(doc_ids)
ax2.set_title("Per-doc Score Heatmap (RdYlGn)")
plt.colorbar(im, ax=ax2)

# Annotate cells
for i in range(len(doc_ids)):
    for j in range(len(all_candidates)):
        v = score_matrix[i, j]
        if not np.isnan(v):
            ax2.text(j, i, f"{v:.2f}", ha="center", va="center",
                     fontsize=7, color="black" if 0.3 < v < 0.8 else "white")

plt.tight_layout()
results_png = BASE_DIR / "Experimentation" / "Ivy" / "gepa_results.png"
plt.savefig(str(results_png), dpi=150)
plt.show()
print(f"Plot saved to {results_png}")

# ── Pareto frontier summary ───────────────────────────────────────────────────
print("\n" + "="*64)
print("PARETO FRONTIER SUMMARY")
print("="*64)
for c in sorted(pareto_frontier, key=lambda x: -x.mean_score):
    print(f"\n[{c.prompt_id}]  iter={c.iteration}  mean={c.mean_score:.3f}  min={c.min_score:.3f}")
    for t in sorted(c.traces, key=lambda x: x.doc_id):
        print(f"  {t.doc_id}: {t.score:.3f}")
    print(f"  Prompt (first 300 chars): {c.system_prompt[:300]}...")

# ── Export artifact ───────────────────────────────────────────────────────────
artifact = {
    "metadata": {
        "n_iterations": N_ITERATIONS,
        "n_docs":       len(PDF_FILES),
        "doc_ids":      sorted(PDF_FILES.keys()),
    },
    "traces": [
        {
            "iteration": t.iteration,
            "doc_id":    t.doc_id,
            "prompt_id": t.prompt_id,
            "score":     t.score,
            "subscores": t.subscores,
            "n_issues":  len(t.issues),
        }
        for t in all_traces
    ],
    "pareto_frontier": [
        {
            "prompt_id":      c.prompt_id,
            "iteration":      c.iteration,
            "mean_score":     c.mean_score,
            "min_score":      c.min_score,
            "system_prompt":  c.system_prompt,
            "per_doc_scores": {t.doc_id: t.score for t in c.traces},
        }
        for c in sorted(pareto_frontier, key=lambda x: -x.mean_score)
    ],
    "iteration_history": iteration_history,
}

artifact_path = BASE_DIR / "Experimentation" / "Ivy" / "gepa_artifact.json"
with open(artifact_path, "w") as f:
    json.dump(artifact, f, indent=2, default=str)
print(f"\nArtifact saved to {artifact_path}")